# Federated Learning Experiments

## Secure Federated Learning with Differential Privacy for Multi-Disease Chest X-ray Classification

This notebook documents the Federated Learning experiments performed for the SIC project.

### Main objectives

- Prepare a multi-label Chest X-ray dataset
- Simulate multiple FL clients
- Train local CNN models
- Perform Federated Averaging (FedAvg)
- Evaluate the global model
- Compare with centralized training
- Perform threshold tuning
- Conduct a Differential Privacy proof-of-concept experiment

> **Important:** The experiments use simulated FL clients. The clients do not represent actual hospitals.

## 1. Experiment Overview

The FL experiment used the NIH ChestX-ray14 dataset.

### Selected target diseases

1. Pneumonia
2. Atelectasis
3. Cardiomegaly
4. Effusion
5. Nodule
6. Infiltration

The task is **multi-label classification**, meaning one X-ray can contain more than one target condition.

### Dataset subset

- Total images: **6,797**
- Unique patients: **4,359**
- Training images: **4,721**
- Validation images: **1,073**
- Test images: **1,003**

The split was performed at the **patient level** to prevent the same patient from appearing in multiple splits.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms

from PIL import Image
from torch.utils.data import Dataset, DataLoader

In [ ]:
TARGET_DISEASES = [
    "Pneumonia",
    "Atelectasis",
    "Cardiomegaly",
    "Effusion",
    "Nodule",
    "Infiltration"
]

NUM_CLIENTS = 5
NUM_ROUNDS = 10
LOCAL_EPOCHS = 1
BATCH_SIZE = 32
LEARNING_RATE = 1e-4

print("Target diseases:", TARGET_DISEASES)
print("Number of clients:", NUM_CLIENTS)
print("FedAvg rounds:", NUM_ROUNDS)

## 2. Dataset Preparation

The metadata was processed to create a six-disease multi-label classification problem.

The selected subset contained up to approximately 1,000 samples per target disease plus No Finding, while preserving multi-label disease overlap.

In [ ]:
DATA_DIR = os.path.join(PROJECT, "data", "chest_xray")

train_csv = os.path.join(DATA_DIR, "train_labeled.csv")
val_csv = os.path.join(DATA_DIR, "val_labeled.csv")
test_csv = os.path.join(DATA_DIR, "test_labeled.csv")

train_df = pd.read_csv(train_csv)
val_df = pd.read_csv(val_csv)
test_df = pd.read_csv(test_csv)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

In [ ]:
print("Train patients:", train_df["Patient ID"].nunique())
print("Validation patients:", val_df["Patient ID"].nunique())
print("Test patients:", test_df["Patient ID"].nunique())

train_patients = set(train_df["Patient ID"])
val_patients = set(val_df["Patient ID"])
test_patients = set(test_df["Patient ID"])

print("Train-Val overlap:", len(train_patients & val_patients))
print("Train-Test overlap:", len(train_patients & test_patients))
print("Val-Test overlap:", len(val_patients & test_patients))

## 3. PyTorch Dataset and Preprocessing

Images were resized to 224 × 224 and normalized using ImageNet normalization because ResNet18 was used as the backbone.

In [ ]:
IMAGE_DIR = os.path.join(DATA_DIR, "images")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


class ChestXrayDataset(Dataset):

    def __init__(self, dataframe, image_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = os.path.join(
            self.image_dir,
            row["Image Index"]
        )

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        labels = torch.tensor(
            row[TARGET_DISEASES].values.astype(float),
            dtype=torch.float32
        )

        return image, labels

In [ ]:
train_dataset = ChestXrayDataset(
    train_df, IMAGE_DIR, transform
)

val_dataset = ChestXrayDataset(
    val_df, IMAGE_DIR, transform
)

test_dataset = ChestXrayDataset(
    test_df, IMAGE_DIR, transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("DataLoaders created.")

## 4. Global Model

A pretrained ResNet18 was used as the CNN backbone.

The final classification layer was replaced with a six-output fully connected layer because the task contains six target diseases.

The loss function was **BCEWithLogitsLoss**, which is appropriate for multi-label classification.

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


def create_model():

    model = models.resnet18(weights="DEFAULT")

    model.fc = nn.Linear(
        model.fc.in_features,
        len(TARGET_DISEASES)
    )

    return model.to(device)


criterion = nn.BCEWithLogitsLoss()

model = create_model()

print(model.fc)

## 5. Simulated FL Client Partition

Five simulated clients were created from the training data.

The partition was performed using a random patient-level split with seed 42.

> **Important:** This particular experiment is a random split and is therefore approximately IID. It should not be described as a true non-IID hospital experiment.

In [ ]:
from sklearn.model_selection import train_test_split

unique_patients = train_df["Patient ID"].unique()

client_patients = np.array_split(
    np.random.RandomState(42).permutation(unique_patients),
    NUM_CLIENTS
)

client_dfs = []

for i, patients in enumerate(client_patients):

    client_df = train_df[
        train_df["Patient ID"].isin(patients)
    ].copy()

    client_dfs.append(client_df)

    print(
        f"Client {i+1}: "
        f"{len(client_df)} images, "
        f"{client_df['Patient ID'].nunique()} patients"
    )

## 6. Local Client Training

Each client trained a copy of the global model locally.

Configuration:

- Optimizer: Adam
- Learning rate: 1e-4
- Local epochs: 1
- Loss: BCEWithLogitsLoss
- Batch size: 32

In [ ]:
def train_local_model(
    model,
    loader,
    epochs=1,
    lr=1e-4
):

    model.train()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    total_loss = 0.0

    for epoch in range(epochs):

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

    return total_loss / len(loader)

## 7. Federated Averaging

After local training, the client model parameters were sent to the server.

The server performed **sample-weighted FedAvg**.

The aggregated global model was then distributed back to the clients for the next round.

In [ ]:
def fedavg(state_dicts, client_sizes):

    total_samples = sum(client_sizes)

    global_state = {}

    for key in state_dicts[0].keys():

        global_state[key] = sum(
            state_dicts[i][key] *
            (client_sizes[i] / total_samples)
            for i in range(len(state_dicts))
        )

    return global_state

## 8. Official FedAvg Experiment — 10 Rounds

The completed experiment used:

- 5 clients
- 10 FedAvg rounds
- 1 local epoch per round
- ResNet18
- BCEWithLogitsLoss

The recorded results were:

| Round | Avg Client Loss | Global Validation Loss |
|------:|----------------:|-----------------------:|
| 1 | 0.6157 | 0.5686 |
| 2 | 0.4782 | 0.4889 |
| 3 | 0.4402 | 0.4788 |
| 4 | 0.4050 | **0.4753** |
| 5 | 0.3660 | 0.4776 |
| 6 | 0.3298 | 0.4851 |
| 7 | 0.2841 | 0.4991 |
| 8 | 0.2454 | 0.5093 |
| 9 | 0.2041 | 0.5245 |
| 10 | 0.1660 | 0.5415 |

### Best round

**Round 4** achieved the lowest global validation loss:

**0.4753**

The model was therefore saved based on validation performance rather than simply taking the final round.

## 9. Saved FedAvg Results

The experiment results and trained model are stored in:

`federated_learning/saved_models/`

In [ ]:
SAVED_DIR = os.path.join(
    PROJECT,
    "federated_learning",
    "saved_models"
)

print("Saved experiment files:")

for file in sorted(os.listdir(SAVED_DIR)):
    print(" -", file)

In [ ]:
history_file = os.path.join(
    SAVED_DIR,
    "fedavg_history.csv"
)

if os.path.exists(history_file):

    history_df = pd.read_csv(history_file)

    print(history_df)

    print("\nColumns:")
    print(history_df.columns.tolist())

## 10. Centralized Training Baseline

A centralized baseline was also trained for comparison.

Recorded training loss over 10 epochs:

| Epoch | Loss |
|------:|-----:|
| 1 | 0.5051 |
| 2 | 0.4011 |
| 3 | 0.2823 |
| 4 | 0.1393 |
| 5 | 0.0594 |
| 6 | 0.0275 |
| 7 | 0.0169 |
| 8 | 0.0127 |
| 9 | 0.0094 |
| 10 | 0.0072 |

The centralized model's training loss decreased substantially during training.

The final centralized model weights were not retained after the runtime disconnect, so final centralized test metrics should **not** be claimed from this experiment.

## 11. FedAvg Test Evaluation

The saved best FedAvg model was evaluated on the held-out test set.

Because this is a multi-label problem, metrics such as F1, precision, recall and ROC-AUC are more informative than strict exact-match accuracy.

### Initial threshold = 0.5

| Metric | Macro Score |
|---|---:|
| F1 | **0.2733** |
| Precision | **0.4945** |
| Recall | **0.1964** |
| ROC-AUC | **0.7032** |

Strict exact-match accuracy was **19.44%**.

This metric is very strict because all six disease labels must be correct for an image to count as an exact match.

### Per-disease results at threshold 0.5

| Disease | F1 | Precision | Recall | ROC-AUC |
|---|---:|---:|---:|---:|
| Pneumonia | 0.0000 | 0.0000 | 0.0000 | 0.5867 |
| Atelectasis | 0.3035 | 0.5234 | 0.2137 | 0.7202 |
| Cardiomegaly | 0.3254 | 0.6939 | 0.2125 | 0.8125 |
| Effusion | 0.5309 | 0.7088 | 0.4243 | 0.8050 |
| Nodule | 0.1577 | 0.5278 | 0.0927 | 0.6591 |
| Infiltration | 0.3223 | 0.5132 | 0.2349 | 0.6359 |

## 12. Validation-Based Threshold Tuning

A single 0.5 threshold was not optimal for every disease.

Disease-specific thresholds were selected using the validation set and then applied to the test set.

This improved the macro F1 and recall substantially.

### Tuned test results

| Metric | Tuned Score |
|---|---:|
| Macro F1 | **0.4580** |
| Macro Precision | **0.3704** |
| Macro Recall | **0.6187** |
| Macro ROC-AUC | **0.7032** |

### Per-disease tuned results

| Disease | F1 | Precision | Recall | ROC-AUC |
|---|---:|---:|---:|---:|
| Pneumonia | 0.2724 | 0.1921 | 0.4679 | 0.5867 |
| Atelectasis | 0.4710 | 0.4365 | 0.5115 | 0.7202 |
| Cardiomegaly | 0.4725 | 0.3732 | 0.6438 | 0.8125 |
| Effusion | 0.6289 | 0.5332 | 0.7664 | 0.8050 |
| Nodule | 0.3975 | 0.2937 | 0.6146 | 0.6591 |
| Infiltration | 0.5059 | 0.3936 | 0.7078 | 0.6359 |

## 13. Differential Privacy Experiment

Differential Privacy was investigated using **Opacus**.

The original ResNet18 model could not be directly validated by Opacus because of compatibility issues involving BatchNorm/in-place operations.

Therefore, a smaller CNN architecture (**DPCNN**) was used for the DP proof-of-concept.

In [ ]:
class DPCNN(nn.Module):

    def __init__(self, num_classes=6):

        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Linear(
            128,
            num_classes
        )

    def forward(self, x):

        x = self.features(x)
        x = torch.flatten(x, 1)

        return self.classifier(x)

### DP configuration

- Target epsilon: **8**
- Delta: **1e-5**
- Maximum gradient norm: **1.0**
- Opacus PrivacyEngine

### One-client DP proof-of-concept

The initial DP experiment successfully trained a client:

- Loss: **0.6914**
- Epsilon: approximately **7.9985**

This confirmed that the DPCNN architecture could be used with Opacus.

## 14. DP-FedAvg Experiment

A 3-round DP-FedAvg experiment was attempted with 5 clients.

Round 1 local results:

| Client | Loss | Epsilon |
|---|---:|---:|
| 1 | 0.6966 | 7.9985 |
| 2 | 0.6965 | 7.9988 |
| 3 | 0.6970 | 7.9969 |
| 4 | 0.6960 | 7.9985 |
| 5 | 0.6954 | 7.9969 |

The aggregation step encountered an Opacus wrapper issue:

`_module.features.0.weight`

instead of:

`features.0.weight`

The local Opacus model therefore needed to be unwrapped before FedAvg aggregation.

## 15. Differential Privacy Limitations

The DP experiment should be described as a **proof-of-concept**, not as a production-grade end-to-end privacy guarantee.

Important limitations:

- Each local training call created a new PrivacyEngine.
- Privacy accounting was therefore not a rigorous composed epsilon across all FedAvg rounds and clients.
- Secure RNG was not enabled.
- Secure aggregation was not implemented.
- The DP experiment used DPCNN rather than the main ResNet18 model.

Therefore, the reported epsilon values should not be presented as the total privacy budget of the complete FL system.

## 16. Saved Experiment Artifacts

The following artifacts were produced during the experiments.

### Trained models

- `fedavg_best_model.pth`
- `dp/dp_fedavg_model.pth`

### FedAvg results

- `fedavg_history.csv`
- `fedavg_validation_metrics.csv`
- `fedavg_test_metrics.csv`
- `fedavg_final_test_metrics.csv`
- `fedavg_macro_metrics.csv`
- `fedavg_tuned_test_metrics.csv`
- `fedavg_tuned_macro_metrics.csv`
- `fedavg_thresholds.csv`
- `client_distribution.csv`
- `fedavg_experiment_summary.csv`

### Baseline

- `centralized_history.csv`

### Visualization

- `fedavg_training_history.png`

## 17. Experiment Conclusion

The completed experiments demonstrate a working federated multi-label chest X-ray classification pipeline using simulated clients and FedAvg.

The best global model was obtained at **FedAvg Round 4**, with a validation loss of **0.4753**.

The original 0.5 classification threshold produced a macro F1 of **0.2733**. Validation-based disease-specific threshold tuning increased the macro F1 to **0.4580** and macro recall to **0.6187** on the test set.

The experiments also explored Differential Privacy using Opacus. A DPCNN-based proof-of-concept successfully trained with a target epsilon of approximately 8, although the complete DP-FedAvg experiment requires more rigorous privacy accounting and engineering before it can be presented as a formal privacy guarantee.

### Important interpretation

The current FL experiment uses **simulated random clients**, so it demonstrates the federated training workflow but does not yet demonstrate performance under a true hospital-level non-IID partition.